# FFT vs PFB Spectrometer Comparison
### QICK RFSoC 4x2 — RHINO Science Band Investigation

This notebook implements and compares two software-defined spectrometers using raw ADC samples from the QICK overlay:

| Parameter | FFT Spectrometer | PFB Spectrometer |
|---|---|---|
| FFT length | 16384 points | 16384 points |
| Window | Hann | Hann-windowed sinc (prototype filter) |
| Taps | 1 | 4 |
| Freq resolution | fs/N | fs/N (same) |
| Stopband rejection | ~31 dB (Hann) | ~60–80 dB |

**Run every cell top to bottom.** Every cell prints ✅ PASS or ❌ FAIL.

---

## Cell 1 — Imports and environment check

In [ ]:
import sys, os, time, datetime
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import pkg_resources

REQUIRED = ['numpy', 'matplotlib', 'scipy', 'qick']
missing  = []
for pkg in REQUIRED:
    try:
        __import__(pkg)
        print(f'   {pkg} importable')
    except ImportError:
        print(f'   {pkg} NOT FOUND')
        missing.append(pkg)

import scipy.signal as signal
import scipy.fft    as sfft

print(f'\n  Python  : {sys.version.split()[0]}')
print(f'  NumPy   : {np.__version__}')
import scipy
print(f'  SciPy   : {scipy.__version__}')

SAVE_PATH = '/home/xilinx/jupyter_notebooks/spectrum-analyzer/'
os.makedirs(SAVE_PATH, exist_ok=True)

if missing:
    print(f'\n FAIL — missing: {missing}')
else:
    print('\n PASS — all imports successful')

## Cell 2 — Spectrometer parameters
Edit these values if needed. All subsequent cells use them automatically.

In [ ]:
# ── Spectrometer configuration ────────────────────────────────────────────
N_FFT         = 16384       # FFT length for both spectrometers
N_TAPS        = 4           # number of PFB taps
WINDOW        = 'hann'      # window function for FFT spectrometer
N_AVG         = 100         # number of spectra to average for comparison plots

# ── Hardware ──────────────────────────────────────────────────────────────
FS_MHZ        = 4423.680    # ADC sample rate — confirmed from print(soc)
ADC_CH        = 0           # readout channel 0 = ADC_D

# ── RHINO science band ────────────────────────────────────────────────────
HI_FREQ_MHZ   = 1420.405    # 21cm hydrogen line rest frequency
RHINO_LO_MHZ  = 1400.0      # RHINO band lower edge
RHINO_HI_MHZ  = 1440.0      # RHINO band upper edge

# ── Computed ──────────────────────────────────────────────────────────────
NYQUIST_MHZ   = FS_MHZ / 2
DF_KHZ        = FS_MHZ * 1e3 / N_FFT
N_BLOCK       = N_FFT * N_TAPS        # total samples needed for one PFB output
DDR4_SAMPLES  = N_BLOCK * N_AVG + 100 # total samples to capture

# Frequency axis (positive frequencies only, rfft output)
freq_axis_mhz = np.fft.rfftfreq(N_FFT, d=1.0/FS_MHZ)
n_bins        = len(freq_axis_mhz)    # N_FFT // 2 + 1

# Find bin indices for RHINO band and HI line
hi_bin    = int(np.argmin(np.abs(freq_axis_mhz - HI_FREQ_MHZ)))
rhino_lo  = int(np.argmin(np.abs(freq_axis_mhz - RHINO_LO_MHZ)))
rhino_hi  = int(np.argmin(np.abs(freq_axis_mhz - RHINO_HI_MHZ)))

print(f'  FFT length       : {N_FFT:,} points')
print(f'  PFB taps         : {N_TAPS}')
print(f'  Freq resolution  : {DF_KHZ:.3f} kHz/bin')
print(f'  Nyquist          : {NYQUIST_MHZ:.3f} MHz')
print(f'  Total bins       : {n_bins:,}')
print(f'  PFB block size   : {N_BLOCK:,} samples ({N_FFT} x {N_TAPS} taps)')
print(f'  Samples to capture: {DDR4_SAMPLES:,}')
print(f'  HI line          : {HI_FREQ_MHZ} MHz -> bin {hi_bin:,}')
print(f'  RHINO band       : {RHINO_LO_MHZ}–{RHINO_HI_MHZ} MHz '
      f'(bins {rhino_lo:,}–{rhino_hi:,})')
print(f'\n✅ PASS — parameters set')

## Cell 3 — PYNQ and QICK version check

In [ ]:
passed = True

try:
    pynq_ver = pkg_resources.get_distribution('pynq').version
    major, minor = int(pynq_ver.split('.')[0]), int(pynq_ver.split('.')[1])
    if major == 3 and minor == 0:
        print(f'   PYNQ {pynq_ver} — QICK compatible')
    else:
        print(f'   PYNQ {pynq_ver} — QICK requires 3.0.x')
        passed = False
except Exception as e:
    print(f'   Cannot read PYNQ version: {e}')
    passed = False

try:
    qick_ver = pkg_resources.get_distribution('qick').version
    print(f'   QICK {qick_ver}')
except Exception as e:
    print(f'   Cannot read QICK version: {e}')
    passed = False

print(f'\n{" PASS" if passed else " FAIL"} — version check')

## Cell 4 — Load QICK overlay
Takes ~60 seconds. The FPGA is being programmed.

In [ ]:
from qick import QickSoc

BIT_FILE = '/home/xilinx/qick_repo/qick_lib/qick/qick_4x2.bit'

if not os.path.exists(BIT_FILE):
    print(f' FAIL — bitstream not found: {BIT_FILE}')
    soc = None
else:
    print('[LOAD] Programming FPGA — please wait ~60s...')
    t0  = time.time()
    soc = QickSoc(bitfile=BIT_FILE)
    print(f'[LOAD] Done in {time.time()-t0:.1f}s')
    print('\n PASS — overlay loaded')
    print(soc)

## Cell 5 — Hardware validation

In [ ]:
if soc is None:
    print(' SKIP — soc not loaded')
else:
    passed = True
    adc_fs  = soc.config['readouts'][0]['fs']
    has_ddr = hasattr(soc, 'arm_ddr4') and hasattr(soc, 'get_ddr4')

    print(f'  ADC fs   : {adc_fs:.3f} MHz', end='')
    if abs(adc_fs - FS_MHZ) < 1.0:
        print('  ✅')
    else:
        print(f'   expected {FS_MHZ}')
        # Update to hardware-reported value
        FS_MHZ       = adc_fs
        NYQUIST_MHZ  = FS_MHZ / 2
        DF_KHZ       = FS_MHZ * 1e3 / N_FFT
        freq_axis_mhz = np.fft.rfftfreq(N_FFT, d=1.0/FS_MHZ)
        n_bins        = len(freq_axis_mhz)
        hi_bin        = int(np.argmin(np.abs(freq_axis_mhz - HI_FREQ_MHZ)))
        rhino_lo      = int(np.argmin(np.abs(freq_axis_mhz - RHINO_LO_MHZ)))
        rhino_hi      = int(np.argmin(np.abs(freq_axis_mhz - RHINO_HI_MHZ)))
        print(f'  [INFO] Updated FS_MHZ to {FS_MHZ:.3f} — parameters recalculated')

    print(f'  DDR4 API : {" available" if has_ddr else " missing"}')
    if not has_ddr:
        passed = False

    print(f'\n{" PASS" if passed else " FAIL"} — hardware validation')

## Cell 6 — Build spectrometer kernels
Constructs the Hann window (FFT) and the PFB prototype filter coefficients (PFB).  
The prototype filter is a Hann-windowed sinc — the standard design for radio astronomy PFBs.

In [ ]:
# ── FFT spectrometer window ───────────────────────────────────────────────
fft_window      = np.hanning(N_FFT).astype(np.float64)
fft_window_norm = np.sum(fft_window**2)   # for power normalisation

print(f'  FFT window     : Hann, N={N_FFT:,}')
print(f'  FFT coherent   : {np.sum(fft_window):.1f}  (sum of window)')
print(f'  FFT power norm : {fft_window_norm:.1f}')

# ── PFB prototype filter ──────────────────────────────────────────────────
# Standard design: Hann-windowed sinc
# Total filter length = N_FFT * N_TAPS
# The sinc is normalised so its passband gain is 1

pfb_len   = N_FFT * N_TAPS

# Sinc function: normalised cutoff at 1/N_FFT (one channel bandwidth)
t         = np.arange(pfb_len) - pfb_len // 2
sinc_arg  = t / N_FFT
proto     = np.sinc(sinc_arg)                       # sinc kernel
hann_full = np.hanning(pfb_len)                     # Hann window over full length
pfb_coeffs = (proto * hann_full).astype(np.float64) # windowed sinc

# Normalise so the sum of all taps for one channel equals 1
pfb_coeffs /= np.sum(pfb_coeffs.reshape(N_TAPS, N_FFT), axis=0).mean()

pfb_window_norm = np.sum(pfb_coeffs**2)

print(f'\n  PFB taps       : {N_TAPS}')
print(f'  PFB filter len : {pfb_len:,} samples')
print(f'  PFB coeff min  : {pfb_coeffs.min():.4f}')
print(f'  PFB coeff max  : {pfb_coeffs.max():.4f}')

print('\n PASS — spectrometer kernels built')

## Cell 7 — Plot filter frequency responses
Shows the stopband rejection of the Hann window (FFT) vs the PFB prototype filter.  
This is the key theoretical comparison — how much leakage each approach produces.

In [ ]:
# Compute frequency responses
NFFT_RESP = 65536   # high resolution for response plot

# FFT window response — pad to NFFT_RESP
fft_padded  = np.zeros(NFFT_RESP)
fft_padded[:N_FFT] = fft_window / fft_window.sum()
fft_resp    = 20 * np.log10(np.abs(np.fft.fft(fft_padded)) + 1e-100)
fft_resp   -= fft_resp.max()   # normalise to 0 dB peak

# PFB prototype filter response
pfb_padded  = np.zeros(NFFT_RESP)
pfb_padded[:pfb_len] = pfb_coeffs / pfb_coeffs.sum()
pfb_resp    = 20 * np.log10(np.abs(np.fft.fft(pfb_padded)) + 1e-100)
pfb_resp   -= pfb_resp.max()

# Frequency axis in normalised units (bins)
bins = np.fft.fftfreq(NFFT_RESP) * NFFT_RESP

# Stopband rejection at ±2 bins separation
idx_2bin    = int(2 * NFFT_RESP / N_FFT)
fft_rej_2   = float(fft_resp[idx_2bin])
pfb_rej_2   = float(pfb_resp[idx_2bin])

print(f'  Leakage at ±2 bin offset:')
print(f'    FFT (Hann)  : {fft_rej_2:.1f} dB')
print(f'    PFB ({N_TAPS} taps): {pfb_rej_2:.1f} dB')
print(f'    Improvement : {pfb_rej_2 - fft_rej_2:.1f} dB from PFB')

# Plot
fig, ax = plt.subplots(figsize=(14, 5))
fig.patch.set_facecolor('#0d0d0d')
ax.set_facecolor('#0d0d0d')

plot_range = 8   # show ±8 bins
mask = (bins >= -plot_range) & (bins <= plot_range)

ax.plot(bins[mask], fft_resp[mask],
        color='#00e5ff', linewidth=1.5,
        label=f'FFT spectrometer (Hann window, 1 tap)')
ax.plot(bins[mask], pfb_resp[mask],
        color='#ff6b35', linewidth=1.5,
        label=f'PFB spectrometer (Hann-sinc, {N_TAPS} taps)')

ax.axhline(-31,  color='#00e5ff', linewidth=0.8,
           linestyle=':', alpha=0.6, label='Hann first sidelobe (~31 dB)')
ax.axhline(pfb_rej_2, color='#ff6b35', linewidth=0.8,
           linestyle=':', alpha=0.6,
           label=f'PFB at ±2 bins ({pfb_rej_2:.0f} dB)')
ax.axvline(-1, color='white', linewidth=0.5, linestyle='--', alpha=0.3)
ax.axvline(+1, color='white', linewidth=0.5, linestyle='--', alpha=0.3)

ax.set_xlim(-plot_range, plot_range)
ax.set_ylim(-120, 5)
ax.set_xlabel('Frequency offset (bins)', color='white', fontsize=12)
ax.set_ylabel('Power response (dB)',      color='white', fontsize=12)
ax.set_title(
    f'Spectrometer channel frequency response comparison\n'
    f'FFT N={N_FFT:,} | PFB N={N_FFT:,} x {N_TAPS} taps | '
    f'Δf = {DF_KHZ:.3f} kHz/bin',
    color='white', fontsize=11)
ax.tick_params(colors='white')
ax.spines[:].set_color('#333333')
ax.grid(True, color='#1e1e1e', linewidth=0.5)
ax.legend(facecolor='#1a1a1a', edgecolor='#444',
          labelcolor='white', fontsize=9)

plt.tight_layout()
out = f'{SAVE_PATH}spectrometer_response_comparison.png'
fig.savefig(out, dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
plt.close(fig)
print(f'\n[PLOT] Saved: {out}')
print('\n PASS — filter response comparison complete')

## Cell 8 — Implement spectrometer functions
Defines `fft_spectrometer()` and `pfb_spectrometer()` — both take a 1D numpy array of real samples and return a power spectrum in dB.

In [ ]:
def fft_spectrometer(samples, n_fft=N_FFT, window=None, window_norm=None):
    """
    FFT spectrometer with Hann window.

    Takes the first n_fft samples, applies the Hann window,
    computes the real FFT, and returns power in dB.

    Parameters
    ----------
    samples     : 1D float array, length >= n_fft
    n_fft       : FFT length
    window      : pre-computed window array (uses fft_window if None)
    window_norm : pre-computed sum(window**2) for normalisation

    Returns
    -------
    power_db : 1D float array, length = n_fft // 2 + 1 (positive freqs)
    """
    if window is None:
        window = fft_window
    if window_norm is None:
        window_norm = fft_window_norm

    block    = samples[:n_fft].astype(np.float64)
    windowed = block * window
    spectrum = np.abs(np.fft.rfft(windowed, n=n_fft))**2
    spectrum /= window_norm
    return 10 * np.log10(spectrum + 1e-100)


def pfb_spectrometer(samples, n_fft=N_FFT, n_taps=N_TAPS, coeffs=None):
    """
    Polyphase Filter Bank spectrometer.

    Applies the prototype filter across n_taps consecutive blocks
    of n_fft samples and computes one output spectrum per call.

    Architecture:
        1. Reshape samples into (n_taps, n_fft) matrix
        2. Multiply each row by the corresponding tap of the
           prototype filter (shape n_taps x n_fft)
        3. Sum across taps (polyphase summation)
        4. Apply FFT to the summed row
        5. Return power spectrum in dB

    Parameters
    ----------
    samples : 1D float array, length >= n_fft * n_taps
    n_fft   : FFT length (number of output channels)
    n_taps  : number of filter taps
    coeffs  : prototype filter coefficients, length = n_fft * n_taps

    Returns
    -------
    power_db : 1D float array, length = n_fft // 2 + 1 (positive freqs)
    """
    if coeffs is None:
        coeffs = pfb_coeffs

    block_len = n_fft * n_taps
    block     = samples[:block_len].astype(np.float64)

    # Reshape into (n_taps, n_fft) and apply prototype filter
    # coeffs is also reshaped: rows are the per-tap filter segments
    data_matrix   = block.reshape(n_taps, n_fft)
    filter_matrix = coeffs.reshape(n_taps, n_fft)

    # Polyphase summation: weight each tap and sum
    weighted_sum  = np.sum(data_matrix * filter_matrix, axis=0)

    # FFT of the polyphase sum
    spectrum      = np.abs(np.fft.rfft(weighted_sum, n=n_fft))**2
    spectrum     /= pfb_window_norm
    return 10 * np.log10(spectrum + 1e-100)


# ── Quick self-test on synthetic data ─────────────────────────────────────
# Inject a single tone at 100 MHz into random noise
fs      = FS_MHZ * 1e6
t_test  = np.arange(N_BLOCK) / fs
noise   = np.random.normal(0, 100, N_BLOCK)
tone    = 5000 * np.sin(2 * np.pi * 100e6 * t_test)   # 100 MHz tone
synth   = noise + tone

fft_out = fft_spectrometer(synth)
pfb_out = pfb_spectrometer(synth)

print(f'  FFT spectrometer output: {len(fft_out)} bins, '
      f'range [{fft_out.min():.1f}, {fft_out.max():.1f}] dB')
print(f'  PFB spectrometer output: {len(pfb_out)} bins, '
      f'range [{pfb_out.min():.1f}, {pfb_out.max():.1f}] dB')

# Both should have the same number of bins
if len(fft_out) == len(pfb_out) == N_FFT // 2 + 1:
    print('\n PASS — both spectrometers return correct output shape')
else:
    print('\n FAIL — output shape mismatch')

## Cell 9 — Synthetic signal validation
Injects a known tone into Gaussian noise and runs both spectrometers.  
This validates the implementations before touching real hardware data.

In [ ]:
np.random.seed(42)
TONE_FREQ_MHZ  = 200.0     # injected tone frequency
TONE_AMPLITUDE = 5000.0    # ADC counts — well above noise
NOISE_RMS      = 100.0     # ADC count noise floor

fs = FS_MHZ * 1e6
t  = np.arange(N_BLOCK) / fs

noise_synth = np.random.normal(0, NOISE_RMS, N_BLOCK)
tone_synth  = TONE_AMPLITUDE * np.sin(2 * np.pi * TONE_FREQ_MHZ * 1e6 * t)
signal_synth = noise_synth + tone_synth

# Run both spectrometers
fft_synth = fft_spectrometer(signal_synth)
pfb_synth = pfb_spectrometer(signal_synth)

# Find the tone bin and measure SNR for each
tone_bin   = int(np.argmin(np.abs(freq_axis_mhz - TONE_FREQ_MHZ)))

fft_peak   = float(fft_synth[tone_bin])
fft_noise  = float(np.median(fft_synth))
fft_snr    = fft_peak - fft_noise

pfb_peak   = float(pfb_synth[tone_bin])
pfb_noise  = float(np.median(pfb_synth))
pfb_snr    = pfb_peak - pfb_noise

print(f'  Injected tone  : {TONE_FREQ_MHZ} MHz -> bin {tone_bin}')
print(f'  FFT peak       : {fft_peak:.1f} dB | noise floor {fft_noise:.1f} dB | SNR {fft_snr:.1f} dB')
print(f'  PFB peak       : {pfb_peak:.1f} dB | noise floor {pfb_noise:.1f} dB | SNR {pfb_snr:.1f} dB')

# Measure leakage at 2 bins either side
fft_leak = float(fft_synth[tone_bin + 2]) - fft_noise
pfb_leak = float(pfb_synth[tone_bin + 2]) - pfb_noise
print(f'\n  Leakage at +2 bins:')
print(f'    FFT : {fft_leak:.1f} dB above noise floor')
print(f'    PFB : {pfb_leak:.1f} dB above noise floor')
print(f'    PFB improvement: {fft_leak - pfb_leak:.1f} dB')

# Plot
ZOOM = 20   # bins either side of tone to show
b0, b1 = max(0, tone_bin - ZOOM), min(n_bins, tone_bin + ZOOM)
freqs_zoom = freq_axis_mhz[b0:b1]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.patch.set_facecolor('#0d0d0d')

for ax, spec, label, colour in [
    (axes[0], fft_synth, f'FFT spectrometer (Hann, 1 tap)', '#00e5ff'),
    (axes[1], pfb_synth, f'PFB spectrometer (Hann-sinc, {N_TAPS} taps)', '#ff6b35'),
]:
    ax.set_facecolor('#0d0d0d')
    ax.plot(freqs_zoom, spec[b0:b1], color=colour, linewidth=1.2)
    ax.axvline(TONE_FREQ_MHZ, color='#88ff88', linewidth=1.0,
               linestyle='--', label=f'Injected tone ({TONE_FREQ_MHZ} MHz)')
    ax.axvline(freq_axis_mhz[tone_bin + 2], color='yellow',
               linewidth=0.8, linestyle=':', alpha=0.7, label='+2 bin offset')
    ax.set_xlabel('Frequency (MHz)', color='white', fontsize=11)
    ax.set_ylabel('Power (dB)',       color='white', fontsize=11)
    ax.set_title(label, color='white', fontsize=10)
    ax.tick_params(colors='white')
    ax.spines[:].set_color('#333333')
    ax.grid(True, color='#1e1e1e', linewidth=0.4)
    ax.legend(facecolor='#1a1a1a', edgecolor='#444',
              labelcolor='white', fontsize=8)

fig.suptitle(
    f'Synthetic tone validation | {TONE_FREQ_MHZ} MHz into Gaussian noise | '
    f'N={N_FFT:,} | {N_TAPS} PFB taps',
    color='white', fontsize=11)
plt.tight_layout()
out = f'{SAVE_PATH}synthetic_tone_validation.png'
fig.savefig(out, dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
plt.close(fig)
print(f'\n[PLOT] Saved: {out}')
print('\n PASS — synthetic validation complete')

## Cell 10 — Synthetic HI line detection test
This is the key scientific validation. Injects a synthetic HI-like tone at **1420.405 MHz**  
alongside a simulated nearby RFI source, then compares how well each spectrometer  
isolates the HI signal from the RFI leakage.

This directly demonstrates the scientific motivation for using a PFB in the RHINO receiver.

In [ ]:
np.random.seed(99)
fs = FS_MHZ * 1e6
t  = np.arange(N_BLOCK) / fs

# Thermal noise floor
noise_floor    = np.random.normal(0, 100.0, N_BLOCK)

# Weak HI-like signal at 1420.405 MHz (amplitude = 500 ADU — 26 dB above noise)
hi_signal      = 500.0 * np.sin(2 * np.pi * HI_FREQ_MHZ * 1e6 * t)

# Strong nearby RFI at 1419.0 MHz (50 MHz offset, amplitude = 20000 ADU — very bright)
# This simulates a real-world RFI scenario near the HI band
RFI_FREQ_MHZ   = 1419.0
rfi_signal     = 20000.0 * np.sin(2 * np.pi * RFI_FREQ_MHZ * 1e6 * t)

combined       = noise_floor + hi_signal + rfi_signal

# Bin indices
hi_bin_test    = int(np.argmin(np.abs(freq_axis_mhz - HI_FREQ_MHZ)))
rfi_bin_test   = int(np.argmin(np.abs(freq_axis_mhz - RFI_FREQ_MHZ)))
sep_bins       = hi_bin_test - rfi_bin_test

print(f'  HI line     : {HI_FREQ_MHZ} MHz -> bin {hi_bin_test}')
print(f'  RFI source  : {RFI_FREQ_MHZ} MHz -> bin {rfi_bin_test}')
print(f'  Separation  : {sep_bins} bins ({abs(HI_FREQ_MHZ - RFI_FREQ_MHZ)*1e3/DF_KHZ:.0f} bins | '
      f'{abs(HI_FREQ_MHZ - RFI_FREQ_MHZ)*1e3:.0f} kHz)')
print(f'  RFI/HI ratio: {20000/500:.0f}x amplitude ({20*np.log10(20000/500):.1f} dB)')

# Run both spectrometers
fft_hi = fft_spectrometer(combined)
pfb_hi = pfb_spectrometer(combined)

# Measure HI SNR for each
noise_ref_bins = list(range(hi_bin_test - 50, hi_bin_test - 10))
fft_noise_ref  = float(np.median(fft_hi[noise_ref_bins]))
pfb_noise_ref  = float(np.median(pfb_hi[noise_ref_bins]))

fft_hi_snr = float(fft_hi[hi_bin_test]) - fft_noise_ref
pfb_hi_snr = float(pfb_hi[hi_bin_test]) - pfb_noise_ref

print(f'\n  HI line recovery:')
print(f'    FFT SNR at HI bin : {fft_hi_snr:.1f} dB above local noise')
print(f'    PFB SNR at HI bin : {pfb_hi_snr:.1f} dB above local noise')

# Plot — zoom into RHINO band
ZOOM_LO = max(0, rfi_bin_test - 30)
ZOOM_HI = min(n_bins, hi_bin_test + 50)
fz = freq_axis_mhz[ZOOM_LO:ZOOM_HI]

fig, axes = plt.subplots(2, 1, figsize=(14, 9))
fig.patch.set_facecolor('#0d0d0d')

for ax, spec, label, colour in [
    (axes[0], fft_hi, f'FFT spectrometer (Hann window, 1 tap)', '#00e5ff'),
    (axes[1], pfb_hi, f'PFB spectrometer (Hann-sinc, {N_TAPS} taps)', '#ff6b35'),
]:
    ax.set_facecolor('#0d0d0d')
    ax.plot(fz, spec[ZOOM_LO:ZOOM_HI], color=colour, linewidth=0.8)
    ax.axvline(HI_FREQ_MHZ, color='#88ff88', linewidth=1.2,
               linestyle='--', label=f'HI line ({HI_FREQ_MHZ} MHz)')
    ax.axvline(RFI_FREQ_MHZ, color='#ff4444', linewidth=1.2,
               linestyle='--', label=f'RFI source ({RFI_FREQ_MHZ} MHz)')
    ax.axvspan(RHINO_LO_MHZ, RHINO_HI_MHZ, alpha=0.06,
               color='#aaffaa', label='RHINO band')
    ax.set_xlabel('Frequency (MHz)', color='white', fontsize=11)
    ax.set_ylabel('Power (dB)',       color='white', fontsize=11)
    ax.set_title(
        f'{label} | HI SNR = {(pfb_hi_snr if "PFB" in label else fft_hi_snr):.1f} dB',
        color='white', fontsize=10)
    ax.tick_params(colors='white')
    ax.spines[:].set_color('#333333')
    ax.grid(True, color='#1e1e1e', linewidth=0.4)
    ax.legend(facecolor='#1a1a1a', edgecolor='#444',
              labelcolor='white', fontsize=9, loc='upper right')

fig.suptitle(
    f'HI line detection: synthetic test | HI={HI_FREQ_MHZ} MHz | '
    f'RFI={RFI_FREQ_MHZ} MHz (40x brighter) | N={N_FFT:,}',
    color='white', fontsize=11)
plt.tight_layout()
out = f'{SAVE_PATH}hi_line_detection_comparison.png'
fig.savefig(out, dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
plt.close(fig)
print(f'\n[PLOT] Saved: {out}')
print('\n PASS — HI detection test complete')

## Cell 11 — Capture real ADC data from QICK DDR4
Captures enough real samples to average `N_AVG` spectra from both spectrometers.  
Antenna should be connected to **ADC_D**.

In [ ]:
from qick.averager_program import AveragerProgram

if soc is None:
    print(' SKIP — soc not loaded')
    raw_samples = None
else:
    class DDR4Trigger(AveragerProgram):
        def initialize(self):
            self.declare_readout(ch=ADC_CH, length=1000, freq=0, gen_ch=None)
            self.synci(200)
        def body(self):
            self.trigger(adcs=[ADC_CH], pins=[0], adc_trig_offset=100)
            self.wait_all()
            self.sync_all(self.us2cycles(1.0))

    prog = DDR4Trigger(soc, {'ro_ch':ADC_CH,'readout_length':1000,
                              'adc_trig_offset':100,'soft_avgs':1,
                              'reps':1,'relax_delay':1.0})

    # Work out DDR4 transfer size
    # Each transfer = 256 samples; capture enough for N_AVG * N_BLOCK samples
    SAMPLES_PER_TRANSFER = 256
    NT = (DDR4_SAMPLES // SAMPLES_PER_TRANSFER) + 10

    print(f'[CAPTURE] Requesting {NT} DDR4 transfers '
          f'(~{NT * SAMPLES_PER_TRANSFER:,} samples)...')
    print(f'[CAPTURE] Need {DDR4_SAMPLES:,} samples for {N_AVG} averaged spectra')

    t0 = time.time()
    try:
        soc.arm_ddr4(ch=ADC_CH, nt=NT)
        soc.run_rounds(prog, rounds=1)
        raw = soc.get_ddr4(ch=ADC_CH, nt=NT, start=None)

        raw_samples = raw[:, 0].astype(np.float32)   # I channel
        elapsed     = time.time() - t0

        clip_frac = float(np.mean(np.abs(raw_samples) > 0.95 * 32767)) * 100

        print(f'\n[CAPTURE] Done in {elapsed:.1f}s')
        print(f'  Samples captured : {len(raw_samples):,}')
        print(f'  I range          : {raw_samples.min():.1f} to {raw_samples.max():.1f} ADU')
        print(f'  RMS              : {float(np.sqrt(np.mean(raw_samples**2))):.2f} ADU')
        print(f'  Clipping         : {clip_frac:.2f}%')

        if clip_frac > 1:
            print('  ⚠️  WARNING — clipping detected, add attenuation')
        if len(raw_samples) < DDR4_SAMPLES:
            print(f'  ⚠️  WARNING — fewer samples than requested '
                  f'({len(raw_samples):,} < {DDR4_SAMPLES:,})')
            print('  Proceeding with available samples')

        print('\n PASS — real ADC data captured')

    except Exception as e:
        print(f'\n FAIL — DDR4 capture error: {e}')
        raw_samples = None

## Cell 12 — Average spectra from real data
Runs both spectrometers over the captured data, averaging `N_AVG` non-overlapping spectra.

In [ ]:
if raw_samples is None:
    print(' SKIP — no raw samples from Cell 11')
    fft_avg = pfb_avg = None
else:
    fft_acc = np.zeros(n_bins, dtype=np.float64)
    pfb_acc = np.zeros(n_bins, dtype=np.float64)
    n_fft_done = 0
    n_pfb_done = 0

    # FFT spectrometer — uses N_FFT samples per spectrum
    max_fft_spectra = len(raw_samples) // N_FFT
    n_fft_use = min(N_AVG, max_fft_spectra)
    for i in range(n_fft_use):
        chunk = raw_samples[i * N_FFT : (i + 1) * N_FFT]
        fft_acc += fft_spectrometer(chunk).astype(np.float64)
        n_fft_done += 1

    # PFB spectrometer — uses N_BLOCK samples per spectrum
    max_pfb_spectra = len(raw_samples) // N_BLOCK
    n_pfb_use = min(N_AVG, max_pfb_spectra)
    for i in range(n_pfb_use):
        chunk = raw_samples[i * N_BLOCK : (i + 1) * N_BLOCK]
        pfb_acc += pfb_spectrometer(chunk).astype(np.float64)
        n_pfb_done += 1

    fft_avg = (fft_acc / n_fft_done).astype(np.float32)
    pfb_avg = (pfb_acc / n_pfb_done).astype(np.float32)

    print(f'  FFT spectra averaged : {n_fft_done}')
    print(f'  PFB spectra averaged : {n_pfb_done}')
    print(f'  FFT mean power       : {fft_avg.mean():.1f} dB')
    print(f'  PFB mean power       : {pfb_avg.mean():.1f} dB')
    print('\n PASS — averaging complete')

## Cell 13 — Full band comparison plot
Side-by-side wideband spectrum from both spectrometers on real ADC data.

In [ ]:
if fft_avg is None or pfb_avg is None:
    print(' SKIP — no averaged spectra from Cell 12')
else:
    ts = datetime.datetime.utcnow().strftime('%Y%m%d_%H%M%S')

    RFI_BANDS = {
        'FM':      (88,   108,  '#aaffaa'),
        'DAB':     (174,  240,  '#aaffaa'),
        '4G':      (700,  960,  '#ffaaff'),
        'GPS':     (1570, 1580, '#aaaaff'),
        'RHINO':   (RHINO_LO_MHZ, RHINO_HI_MHZ, '#ffffaa'),
    }

    fig, axes = plt.subplots(2, 1, figsize=(16, 10))
    fig.patch.set_facecolor('#0d0d0d')

    for ax, spec, label, colour in [
        (axes[0], fft_avg, f'FFT Spectrometer | Hann | N={N_FFT:,} | {DF_KHZ:.2f} kHz/bin', '#00e5ff'),
        (axes[1], pfb_avg, f'PFB Spectrometer | Hann-sinc | N={N_FFT:,} x {N_TAPS} taps | {DF_KHZ:.2f} kHz/bin', '#ff6b35'),
    ]:
        ax.set_facecolor('#0d0d0d')
        ax.plot(freq_axis_mhz, spec, color=colour, linewidth=0.5)
        for name, (f0, f1, col) in RFI_BANDS.items():
            ax.axvspan(f0, f1, alpha=0.10 if name=='RHINO' else 0.06,
                       color=col)
            ax.text((f0+f1)/2, spec.max()+1, name,
                    color=col, fontsize=6, ha='center', va='bottom')
        ax.axvline(HI_FREQ_MHZ, color='#88ff88', linewidth=0.8,
                   linestyle='--', alpha=0.8, label=f'HI line ({HI_FREQ_MHZ} MHz)')
        ax.set_xlim(0, NYQUIST_MHZ)
        ax.set_xlabel('Frequency (MHz)', color='white', fontsize=11)
        ax.set_ylabel('Power (dB)',       color='white', fontsize=11)
        ax.set_title(label, color='white', fontsize=10)
        ax.tick_params(colors='white')
        ax.spines[:].set_color('#333333')
        ax.grid(True, color='#1e1e1e', linewidth=0.3)
        ax.xaxis.set_major_locator(ticker.MultipleLocator(200))
        ax.legend(facecolor='#1a1a1a', edgecolor='#444',
                  labelcolor='white', fontsize=8)

    fig.suptitle(
        f'FFT vs PFB Spectrometer | Real ADC data | RFSoC 4x2 | UTC {ts} | '
        f'{n_fft_done} FFT / {n_pfb_done} PFB spectra averaged',
        color='white', fontsize=10)
    plt.tight_layout()
    out = f'{SAVE_PATH}fft_vs_pfb_fullband_{ts}.png'
    fig.savefig(out, dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
    plt.close(fig)
    print(f'[PLOT] Saved: {out}')
    print('\n PASS — full band comparison plot saved')

## Cell 14 — RHINO science band zoom
Zooms into the 1400–1440 MHz band around the HI line for both spectrometers.

In [ ]:
if fft_avg is None or pfb_avg is None:
    print(' SKIP — no averaged spectra')
else:
    ts  = datetime.datetime.utcnow().strftime('%Y%m%d_%H%M%S')
    fz  = freq_axis_mhz[rhino_lo:rhino_hi]

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    fig.patch.set_facecolor('#0d0d0d')

    for ax, spec, label, colour in [
        (axes[0], fft_avg, f'FFT (Hann, 1 tap)', '#00e5ff'),
        (axes[1], pfb_avg, f'PFB (Hann-sinc, {N_TAPS} taps)', '#ff6b35'),
    ]:
        ax.set_facecolor('#0d0d0d')
        ax.plot(fz, spec[rhino_lo:rhino_hi], color=colour, linewidth=0.8)
        ax.axvline(HI_FREQ_MHZ, color='#88ff88', linewidth=1.2,
                   linestyle='--', label=f'HI line 1420.405 MHz')
        ax.axvspan(RHINO_LO_MHZ, RHINO_HI_MHZ, alpha=0.06,
                   color='#ffffaa', label='RHINO band')
        ax.set_xlabel('Frequency (MHz)', color='white', fontsize=11)
        ax.set_ylabel('Power (dB)',       color='white', fontsize=11)
        ax.set_title(label, color='white', fontsize=11)
        ax.tick_params(colors='white')
        ax.spines[:].set_color('#333333')
        ax.grid(True, color='#1e1e1e', linewidth=0.4)
        ax.xaxis.set_major_locator(ticker.MultipleLocator(5))
        ax.legend(facecolor='#1a1a1a', edgecolor='#444',
                  labelcolor='white', fontsize=9)

    fig.suptitle(
        f'RHINO science band zoom | {RHINO_LO_MHZ}–{RHINO_HI_MHZ} MHz | '
        f'Δf = {DF_KHZ:.3f} kHz/bin | UTC {ts}',
        color='white', fontsize=11)
    plt.tight_layout()
    out = f'{SAVE_PATH}rhino_band_zoom_{ts}.png'
    fig.savefig(out, dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
    plt.close(fig)
    print(f'[PLOT] Saved: {out}')
    print('\n PASS — RHINO band zoom saved')

## Cell 15 — Quantitative comparison summary
Computes key metrics for both spectrometers and prints a summary table  
suitable for including in your thesis.

In [ ]:
if fft_avg is None or pfb_avg is None:
    print(' SKIP — no averaged spectra')
else:
    # Noise floor — use median of full band as reference
    fft_noise_floor = float(np.median(fft_avg))
    pfb_noise_floor = float(np.median(pfb_avg))

    # Dynamic range — peak minus noise floor
    fft_dyn_range   = float(fft_avg.max()) - fft_noise_floor
    pfb_dyn_range   = float(pfb_avg.max()) - pfb_noise_floor

    # Spectral smoothness in RHINO band — std dev of spectrum in quiet region
    # Use 1400-1420 MHz (below HI line, should be relatively quiet)
    quiet_lo = int(np.argmin(np.abs(freq_axis_mhz - 1400.0)))
    quiet_hi = int(np.argmin(np.abs(freq_axis_mhz - 1415.0)))
    fft_rhino_std = float(np.std(fft_avg[quiet_lo:quiet_hi]))
    pfb_rhino_std = float(np.std(pfb_avg[quiet_lo:quiet_hi]))

    # Theoretical stopband at 2 bins offset (from filter response)
    fft_stop_2bin  = fft_rej_2
    pfb_stop_2bin  = pfb_rej_2

    print('=' * 62)
    print('  SPECTROMETER COMPARISON SUMMARY')
    print(f'  {datetime.datetime.utcnow().strftime("%Y-%m-%d %H:%M UTC")}')
    print('=' * 62)
    print(f'  {"Parameter":<30} {"FFT":>12} {"PFB":>12}')
    print('-' * 62)
    print(f'  {"FFT length (points)":<30} {N_FFT:>12,} {N_FFT:>12,}')
    print(f'  {"Filter taps":<30} {1:>12} {N_TAPS:>12}')
    print(f'  {"Window":<30} {"Hann":>12} {"Hann-sinc":>12}')
    print(f'  {"Freq resolution (kHz/bin)":<30} {DF_KHZ:>12.3f} {DF_KHZ:>12.3f}')
    print(f'  {"Stopband at +2 bins (dB)":<30} {fft_stop_2bin:>12.1f} {pfb_stop_2bin:>12.1f}')
    print(f'  {"PFB leakage improvement (dB)":<30} {"-":>12} {pfb_stop_2bin - fft_stop_2bin:>12.1f}')
    print('-' * 62)
    print(f'  {"--- Real data metrics ---"}')
    print(f'  {"Noise floor (dB, median)":<30} {fft_noise_floor:>12.1f} {pfb_noise_floor:>12.1f}')
    print(f'  {"Dynamic range (dB)":<30} {fft_dyn_range:>12.1f} {pfb_dyn_range:>12.1f}')
    print(f'  {"RHINO band std dev (dB)":<30} {fft_rhino_std:>12.3f} {pfb_rhino_std:>12.3f}')
    print(f'  {"Spectra averaged":<30} {n_fft_done:>12} {n_pfb_done:>12}')
    print('-' * 62)
    print(f'  {"--- Computational cost ---"}')
    print(f'  {"Samples per spectrum":<30} {N_FFT:>12,} {N_BLOCK:>12,}')
    print(f'  {"Relative cost (taps)":<30} {"1x":>12} {str(N_TAPS)+"x":>12}')
    print('=' * 62)

    print('\n PASS — summary complete')
    print('\nSave this output for your thesis write-up.')

## Cell 16 — Save all results
Saves numpy arrays for both spectrometers so you can reload and replot later.

In [ ]:
if fft_avg is None or pfb_avg is None:
    print(' SKIP — no data to save')
else:
    ts = datetime.datetime.utcnow().strftime('%Y%m%d_%H%M%S')

    np.save(f'{SAVE_PATH}fft_spectrum_{ts}.npy',  fft_avg)
    np.save(f'{SAVE_PATH}pfb_spectrum_{ts}.npy',  pfb_avg)
    np.save(f'{SAVE_PATH}freq_axis_{ts}.npy',     freq_axis_mhz)
    if raw_samples is not None:
        np.save(f'{SAVE_PATH}raw_samples_{ts}.npy', raw_samples)

    print(f'  Saved to: {SAVE_PATH}')
    print(f'  fft_spectrum_{ts}.npy')
    print(f'  pfb_spectrum_{ts}.npy')
    print(f'  freq_axis_{ts}.npy')
    if raw_samples is not None:
        print(f'  raw_samples_{ts}.npy')

    print(f'\nDownload with:')
    print(f'  scp xilinx@192.168.3.1:{SAVE_PATH}*_{ts}.npy .')
    print(f'  scp xilinx@192.168.3.1:{SAVE_PATH}*_{ts}.png .')

    print('\n PASS — all results saved')